<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 15px; text-align: center; margin-bottom: 20px;">
<h1 style="color: #e94560; font-size: 2.5em; margin: 0;">⚡ Support Vector Machines</h1>
<h2 style="color: #f5f5f5; font-size: 1.4em; margin: 10px 0;">The Complete Master Lecture</h2>
<p style="color: #a8b2d8; font-size: 1em; margin: 5px 0;">From Zero to Expert — No Assumptions. Just Clear Explanations + Visuals.</p>
</div>

---

## 🗺️ Full Roadmap

| # | Topic | What You'll Learn |
|---|-------|------------------|
| 1 | The Big Picture | What problem does SVM solve? |
| 2 | Hyperplanes | What is a decision boundary? |
| 3 | The Margin | Why maximizing the gap matters |
| 4 | Support Vectors | The "special" data points |
| 5 | Hard vs Soft Margin | Real-world messy data |
| 6 | The C Parameter | Controlling strictness (explained from scratch) |
| 7 | The Kernel Trick | Handling non-linear data |
| 8 | Linear Kernel | Formula + visual + when to use |
| 9 | RBF / Gaussian Kernel | Formula + visual + when to use |
| 10 | Polynomial Kernel | Formula + visual + when to use |
| 11 | Sigmoid Kernel | Formula + visual + when to use |
| 12 | Gamma Parameter | Explained from scratch |
| 13 | Full Real-World Example | Breast Cancer dataset |
| 14 | Hyperparameter Tuning | GridSearch + visualizations |
| 15 | SVM for Multi-Class | How it handles 3+ classes |
| 16 | SVM Regression (SVR) | Using SVM to predict numbers |
| 17 | Pros & Cons | With real examples |
| 18 | Cheat Sheet | Quick reference card |

---
> ### 📌 How to use this notebook
> Run cells **one at a time**, top to bottom. Read the markdown explanations **before** running each code cell. Every concept is explained **before** it is used in code.
> 
> No prior ML knowledge assumed. Let's go! 🚀

---
## 📦 Setup — Importing Everything We Need

In [ ]:
# ─── Core numerical & plotting ────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Scikit-learn: datasets ───────────────────────────────────────────────────
from sklearn import datasets
from sklearn.datasets import make_classification, make_circles, make_moons, make_blobs

# ─── Scikit-learn: SVM ────────────────────────────────────────────────────────
from sklearn.svm import SVC, SVR, LinearSVC

# ─── Scikit-learn: utilities ──────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.inspection import DecisionBoundaryDisplay

# ─── Styling ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 110,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.size'        : 11,
})
COLORS = ['#4C72B0', '#DD8452']   # blue, orange
CMAP   = ListedColormap(['#AEC6E8', '#F5C5A3'])

print("✅ Everything imported successfully!")
print("You are ready to learn SVMs from scratch.")

---
# 🧠 Part 1 — The Big Picture: What Problem Does SVM Solve?

### Imagine a simple scenario:

You have a basket of **apples** 🍎 and **oranges** 🍊. Each fruit has two measurements:
- **Weight** (x-axis)
- **Color score** (y-axis)

Your goal: Given a **new fruit**, decide if it's an apple or orange.

This is a **classification problem** — the most common use of SVM.

### The Challenge
You need to draw a **boundary line** that separates the two groups. But:
- Many lines could work
- Which one is the *best* line?

**SVM's answer:** The best line is the one with the **maximum gap** between the two classes.

Let's see this visually:

In [ ]:
# ─── Create simple toy data ───────────────────────────────────────────────────
np.random.seed(0)
n = 20
X_a = np.random.randn(n, 2) + np.array([1.5, 1.5])   # Apples  (cluster 1)
X_b = np.random.randn(n, 2) + np.array([4.5, 4.5])   # Oranges (cluster 2)
X_toy = np.vstack([X_a, X_b])
y_toy = np.array([0]*n + [1]*n)

# ─── Show the problem: many possible lines ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Show data + multiple possible lines
ax = axes[0]
ax.scatter(X_a[:, 0], X_a[:, 1], c=COLORS[0], s=80, edgecolors='k', label='Apple 🍎', zorder=3)
ax.scatter(X_b[:, 0], X_b[:, 1], c=COLORS[1], s=80, edgecolors='k', label='Orange 🍊', zorder=3)
x_range = np.linspace(-1, 8, 200)
lines = [(-1, 5.5, 'red', 'Line A (poor)'),
         (-0.7, 4.5, 'green', 'Line B (ok)'),
         (-1, 4.8, 'purple', 'Line C (ok)')]
for slope, intercept, color, label in lines:
    ax.plot(x_range, slope * x_range + intercept, color=color, lw=2, label=label, alpha=0.7)
ax.set_xlim(-1, 8); ax.set_ylim(-1, 8)
ax.set_title('😕 Many lines could work...\nWhich one is BEST?', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.set_xlabel('Weight'); ax.set_ylabel('Color Score')

# Right: SVM's answer — the maximum margin line
ax = axes[1]
svm_toy = SVC(kernel='linear', C=1e6)
svm_toy.fit(X_toy, y_toy)

xx, yy = np.meshgrid(np.linspace(-1, 8, 300), np.linspace(-1, 8, 300))
Z = svm_toy.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
ax.contour(xx, yy, Z, colors='black', linewidths=2)

Z2 = svm_toy.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contour(xx, yy, Z2, levels=[-1, 1], linestyles='--', colors=[COLORS[0], COLORS[1]], linewidths=1.5)

ax.scatter(X_a[:, 0], X_a[:, 1], c=COLORS[0], s=80, edgecolors='k', label='Apple 🍎', zorder=3)
ax.scatter(X_b[:, 0], X_b[:, 1], c=COLORS[1], s=80, edgecolors='k', label='Orange 🍊', zorder=3)
ax.scatter(svm_toy.support_vectors_[:, 0], svm_toy.support_vectors_[:, 1],
           s=250, facecolors='none', edgecolors='red', linewidths=2.5, zorder=5, label='Support Vectors')

ax.annotate('← Maximum\n   Margin →', xy=(3.0, 2.8), fontsize=11, color='darkgreen', fontweight='bold')
ax.set_xlim(-1, 8); ax.set_ylim(-1, 8)
ax.set_title('✅ SVM picks the line with\nthe MAXIMUM MARGIN', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.set_xlabel('Weight'); ax.set_ylabel('Color Score')

plt.suptitle('The Core Idea of SVM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📐 Part 2 — Hyperplanes: What is a Decision Boundary?

The word **"hyperplane"** sounds scary. It's not.

| Number of Features | Hyperplane is... | Example |
|--------------------|------------------|---------|
| 2 features (2D) | A **line** | `ax + by + c = 0` |
| 3 features (3D) | A **flat plane** | `ax + by + cz + d = 0` |
| 4+ features (nD) | A **hyperplane** | Same idea, more dimensions |

**In plain English:** A hyperplane is just the thing that **divides your data space in two halves**.

When you have 2 features (which is easy to visualize), it's just a **straight line**.

SVM finds the **hyperplane** that **maximizes the margin** between two classes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── 2D: Line ─────────────────────────────────────────────────────────────────
ax = axes[0]
x_line = np.linspace(-3, 3, 100)
ax.plot(x_line, -x_line, 'k-', lw=2.5, label='Decision boundary (line)')
ax.fill_between(x_line, -x_line, 4, alpha=0.2, color=COLORS[1])
ax.fill_between(x_line, -x_line, -4, alpha=0.2, color=COLORS[0])
ax.scatter([1, 1.5, 2], [-2, -2.5, -1.5], c=COLORS[0], s=80, edgecolors='k', zorder=3)
ax.scatter([-1, -1.5, -2], [2, 1.5, 2.5], c=COLORS[1], s=80, edgecolors='k', zorder=3)
ax.set_xlim(-3, 3); ax.set_ylim(-4, 4)
ax.set_title('2D Data\n→ Line separates it', fontweight='bold')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend(fontsize=8)

# ── 3D concept ────────────────────────────────────────────────────────────────
ax = axes[1]
ax.text(0.5, 0.5,
        '3D Data\n→ Flat PLANE\nseparates it\n\n'
        'Like cutting a sandwich\nin half with a flat cut!\n🥪',
        ha='center', va='center', fontsize=13, fontweight='bold',
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.8', facecolor='#f0f4ff', edgecolor='#4C72B0', lw=2))
ax.axis('off')
ax.set_title('3D Data\n→ Plane separates it', fontweight='bold')

# ── nD concept ────────────────────────────────────────────────────────────────
ax = axes[2]
ax.text(0.5, 0.5,
        'N-Dimensional Data\n→ HYPERPLANE\nseparates it\n\n'
        'Same idea, just in\nhigher dimensions.\n'
        'We can\'t visualize it,\nbut math handles it! 🧮',
        ha='center', va='center', fontsize=12, fontweight='bold',
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.8', facecolor='#fff0f0', edgecolor='#DD8452', lw=2))
ax.axis('off')
ax.set_title('N-D Data\n→ Hyperplane separates it', fontweight='bold')

plt.suptitle('What is a Hyperplane?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📏 Part 3 — The Margin: Why Does It Matter?

The **margin** is the **total width of the empty space** between the two classes, measured from the decision boundary to the nearest points of each class.

```
   ● ● ●  |←–– margin ––→|  ● ● ●
          |                |
     margin boundary    margin boundary
                  |
           Decision Boundary
```

### Why maximize the margin?

Think of it like parking a car between two walls:
- If you hug the left wall, a slight error → you hit it
- If you're **right in the center** → you have the most tolerance for error

**Larger margin = more tolerance = better generalization to new data**

A model that barely squeezes between the training points is **overfit** — it learned the training data too specifically. One with a large margin is more **robust**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def draw_margin_illustration(ax, title, noise_level, C_val):
    np.random.seed(1)
    n = 25
    X1 = np.random.randn(n, 2)*0.6 + [1, 1]
    X2 = np.random.randn(n, 2)*0.6 + [3.5, 3.5]
    X  = np.vstack([X1, X2])
    y  = np.array([0]*n + [1]*n)

    clf = SVC(kernel='linear', C=C_val)
    clf.fit(X, y)

    xx, yy = np.meshgrid(np.linspace(-1, 6, 300), np.linspace(-1, 6, 300))
    Z  = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    Z2 = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.15, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='black', linewidths=2.5)
    c1 = ax.contour(xx, yy, Z2, levels=[-1], linestyles='--', colors=COLORS[0], linewidths=2)
    c2 = ax.contour(xx, yy, Z2, levels=[1],  linestyles='--', colors=COLORS[1], linewidths=2)

    ax.scatter(X1[:, 0], X1[:, 1], c=COLORS[0], s=60, edgecolors='k', label='Class A', zorder=3)
    ax.scatter(X2[:, 0], X2[:, 1], c=COLORS[1], s=60, edgecolors='k', label='Class B', zorder=3)
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=200, facecolors='none', edgecolors='red', linewidths=2.5, zorder=5, label='Support Vectors')

    # Draw margin width arrow
    w = clf.coef_[0]
    margin_width = 2 / np.linalg.norm(w)
    ax.set_title(f'{title}\nMargin width ≈ {margin_width:.2f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(-1, 6); ax.set_ylim(-1, 6)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
    return margin_width

m1 = draw_margin_illustration(axes[0], '✅ Large Margin (Good!)', 0.5, 0.1)
m2 = draw_margin_illustration(axes[1], '❌ Small Margin (Risky!)', 0.5, 1000)

plt.suptitle('Large Margin vs Small Margin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Large margin model width : {m1:.3f}")
print(f"Small margin model width : {m2:.3f}")
print()
print("💡 The large margin model is more ROBUST — small shifts in new data won't cause misclassification.")
print("   The small margin model is BRITTLE — data close to the boundary will easily be misclassified.")

---
# 🤝 Part 3.5 — Similarity: The Secret Behind Support Vectors

Before we look at support vectors, we need to answer one question:

> **How does SVM decide which points "matter"?**

The answer is **similarity** — specifically, how similar points from *opposite classes* are to each other.

---

### What does "similar" mean here?

Two data points are **similar** if they are **close together** in the feature space.

We measure this with **Euclidean distance** — the straight-line distance between two points:

$$\text{distance}(A, B) = \sqrt{(x_A - x_B)^2 + (y_A - y_B)^2}$$

- **Small distance** → points are **very similar** (close together)
- **Large distance** → points are **very different** (far apart)

---

### Why does this matter for SVM?

Think about what makes classification *hard*:

- A **blue** point that is far from all **red** points? Easy to classify — no problem.
- A **blue** point that is sitting right next to a **red** point? *Very hard* — they are similar but opposite classes.

**SVM focuses all its attention on those hard cases** — the cross-class pairs that are most similar to each other.

Those are the points that will become **support vectors**.

> 💡 **One sentence summary:**  
> SVM finds the two most similar points from **opposite classes** and places the boundary *exactly between them*.

In [ ]:
# ── Show cross-class similarity: who is closest to whom? ─────────────────────
from sklearn.metrics import pairwise_distances

np.random.seed(7)
blue_pts = np.array([[1.0,2.0],[1.5,3.5],[0.5,1.5],[2.0,2.5],[0.8,3.0]])
red_pts  = np.array([[4.0,3.0],[3.5,1.5],[5.0,2.5],[4.5,4.0],[3.2,2.8]])

# Compute all pairwise distances between the two classes
all_dists = pairwise_distances(blue_pts, red_pts)

# Find the closest pair
b_idx, r_idx = np.unravel_index(np.argmin(all_dists), all_dists.shape)
min_dist = all_dists[b_idx, r_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Left: scatter with all cross-class lines ─────────────────────────────────
ax = axes[0]
for i, bp in enumerate(blue_pts):
    for j, rp in enumerate(red_pts):
        is_closest = (i == b_idx and j == r_idx)
        ax.plot([bp[0], rp[0]], [bp[1], rp[1]],
                color='red' if is_closest else 'gray',
                lw=3   if is_closest else 0.5,
                alpha=1.0 if is_closest else 0.2, zorder=5 if is_closest else 1)
        if is_closest:
            mid = (bp + rp) / 2
            ax.annotate(f'Closest pair!\ndist = {min_dist:.2f}', xy=mid,
                        xytext=(mid[0]-1.5, mid[1]+0.6), fontsize=10,
                        color='red', fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color='red'),
                        bbox=dict(boxstyle='round', facecolor='#fff0f0', edgecolor='red'))

ax.scatter(blue_pts[:,0], blue_pts[:,1], c='#4C72B0', s=120, edgecolors='k',
           zorder=6, label='Class Blue')
ax.scatter(red_pts[:,0],  red_pts[:,1],  c='#DD8452', s=120, edgecolors='k',
           zorder=6, label='Class Red')
# Highlight the closest pair with stars
ax.scatter(*blue_pts[b_idx], c='yellow', s=400, marker='*',
           edgecolors='red', lw=3, zorder=7, label='Future Support Vectors ⭐')
ax.scatter(*red_pts[r_idx],  c='yellow', s=400, marker='*',
           edgecolors='red', lw=3, zorder=7)
ax.set_title('Every cross-class distance\nClosest pair = future Support Vectors',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

# ── Right: distance heatmap ───────────────────────────────────────────────────
ax = axes[1]
sns.heatmap(all_dists, annot=True, fmt='.2f', cmap='RdYlGn_r',
            xticklabels=[f'Red-{i}' for i in range(len(red_pts))],
            yticklabels=[f'Blue-{i}' for i in range(len(blue_pts))],
            ax=ax, linewidths=0.5)
ax.add_patch(plt.Rectangle([r_idx, b_idx], 1, 1,
             fill=False, edgecolor='red', lw=4, zorder=5))
ax.set_title('Distance Heatmap (Blue vs Red)\nRed box = most similar pair across classes',
             fontweight='bold')
ax.set_xlabel('Red points'); ax.set_ylabel('Blue points')

plt.suptitle('Step 1: Find the Most Similar Points from Opposite Classes',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Most similar (closest) cross-class pair:')
print(f'  Blue-{b_idx} at {blue_pts[b_idx]}  and  Red-{r_idx} at {red_pts[r_idx]}')
print(f'  Distance = {min_dist:.3f}')
print()
print('These two points are the hardest to separate.')
print('SVM will place the boundary BETWEEN them — those are the support vectors.')


---
### 🏗️ How Is the Margin Actually Drawn?

Now that we understand *which* points matter (the closest cross-class pair), let's see *exactly* how the margin gets drawn around them.

Here is the step-by-step process:

```
Step 1:  Find the closest Blue point and the closest Red point to each other.

Step 2:  Draw a line through each of those two points, parallel to what will
         become the decision boundary. These are the MARGIN LINES (dashed).

Step 3:  The DECISION BOUNDARY goes exactly HALFWAY between the two margin lines.
         It is equidistant from both support vectors.

Step 4:  The MARGIN WIDTH = total distance between the two margin lines.
         SVM maximizes this width.
```

A good way to picture it:

> Imagine the two support vectors are two people standing facing each other.
> SVM draws a line on the ground exactly **halfway between their feet**.
> Then it pushes both people as far apart as possible while keeping everyone else on the right side.

The **margin** is the gap between their feet. The **boundary** is the line on the ground between them.

In [ ]:
# ── Step-by-step margin drawing ───────────────────────────────────────────────
np.random.seed(0)
blue_m = np.array([[1.0,2.0],[0.5,3.2],[1.3,1.2],[0.7,2.8],[0.3,1.8]])
red_m  = np.array([[3.8,2.5],[4.3,1.5],[3.5,3.5],[4.0,0.8],[4.5,3.0]])
X_m = np.vstack([blue_m, red_m])
y_m = np.array([0]*5 + [1]*5)

clf_m = SVC(kernel='linear', C=1e6)
clf_m.fit(X_m, y_m)
svs   = clf_m.support_vectors_
sv_y  = y_m[clf_m.support_]

sv_blue = svs[sv_y == 0][0]
sv_red  = svs[sv_y == 1][0]
midpoint = (sv_blue + sv_red) / 2

xx_m, yy_m = np.meshgrid(np.linspace(-0.5, 6, 300), np.linspace(-0.5, 5, 300))
Z_m  = clf_m.predict(np.c_[xx_m.ravel(), yy_m.ravel()]).reshape(xx_m.shape)
Z2_m = clf_m.decision_function(np.c_[xx_m.ravel(), yy_m.ravel()]).reshape(xx_m.shape)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

def base(ax):
    ax.scatter(blue_m[:,0], blue_m[:,1], c='#4C72B0', s=100, edgecolors='k', zorder=4, label='Blue')
    ax.scatter(red_m[:,0],  red_m[:,1],  c='#DD8452', s=100, edgecolors='k', zorder=4, label='Red')
    ax.set_xlim(-0.5, 6); ax.set_ylim(-0.5, 5)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

# ── Step 1: just the data ─────────────────────────────────────────────────────
ax = axes[0]
base(ax)
# Connect all cross-class pairs faintly
for bp in blue_m:
    for rp in red_m:
        ax.plot([bp[0],rp[0]],[bp[1],rp[1]], color='gray', lw=0.4, alpha=0.25)
# Closest pair
dists_m = pairwise_distances(blue_m, red_m)
bi_m, ri_m = np.unravel_index(np.argmin(dists_m), dists_m.shape)
ax.plot([blue_m[bi_m,0],red_m[ri_m,0]],
        [blue_m[bi_m,1],red_m[ri_m,1]], 'r-', lw=2.5, zorder=5, label='Closest pair')
ax.scatter(*blue_m[bi_m], c='yellow', s=350, marker='*', edgecolors='red', lw=2.5, zorder=6)
ax.scatter(*red_m[ri_m],  c='yellow', s=350, marker='*', edgecolors='red', lw=2.5, zorder=6)
ax.set_title('Step 1\nFind closest pair\nacross classes', fontweight='bold')
ax.legend(fontsize=8)

# ── Step 2: Draw margin lines through the SVs ─────────────────────────────────
ax = axes[1]
base(ax)
ax.contour(xx_m, yy_m, Z2_m, levels=[-1], linestyles='--', colors='#4C72B0', linewidths=2.5)
ax.contour(xx_m, yy_m, Z2_m, levels=[1],  linestyles='--', colors='#DD8452', linewidths=2.5)
ax.scatter(svs[:,0], svs[:,1], s=300, facecolors='none',
           edgecolors='red', lw=3, zorder=6, label='Support Vectors')
ax.annotate('Blue margin line\n(passes through\nBlue SV)', xy=sv_blue,
            xytext=(sv_blue[0]-1.5, sv_blue[1]+0.6), fontsize=8, color='#4C72B0',
            fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#4C72B0'))
ax.annotate('Red margin line\n(passes through\nRed SV)', xy=sv_red,
            xytext=(sv_red[0]+0.2, sv_red[1]+0.6), fontsize=8, color='#DD8452',
            fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#DD8452'))
ax.set_title('Step 2\nDraw margin lines\nthrough each SV', fontweight='bold')
ax.legend(fontsize=8)

# ── Step 3: Draw boundary exactly halfway ─────────────────────────────────────
ax = axes[2]
base(ax)
ax.contourf(xx_m, yy_m, Z_m, alpha=0.15, cmap=ListedColormap(['#C9DDF5','#FAD7B5']))
ax.contour(xx_m, yy_m, Z_m,  colors='black', linewidths=3)
ax.contour(xx_m, yy_m, Z2_m, levels=[-1,1], linestyles='--',
           colors=['#4C72B0','#DD8452'], linewidths=2)
ax.scatter(svs[:,0], svs[:,1], s=300, facecolors='none',
           edgecolors='red', lw=3, zorder=6, label='Support Vectors')
ax.scatter(*midpoint, c='green', s=250, marker='D', zorder=8,
           edgecolors='darkgreen', lw=2, label='Midpoint (on boundary)')
ax.annotate('Decision boundary\ngoes here — exactly\nhalfway between SVs',
            xy=midpoint, xytext=(midpoint[0]+0.5, midpoint[1]-1.2),
            fontsize=8.5, color='black', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
ax.set_title('Step 3\nBoundary goes exactly\nhalfway between SVs', fontweight='bold')
ax.legend(fontsize=8)

# ── Step 4: Label the margin width ───────────────────────────────────────────
ax = axes[3]
base(ax)
ax.contourf(xx_m, yy_m, Z_m, alpha=0.15, cmap=ListedColormap(['#C9DDF5','#FAD7B5']))
ax.contour(xx_m, yy_m, Z_m,  colors='black', linewidths=3)
ax.contour(xx_m, yy_m, Z2_m, levels=[-1,1], linestyles='--',
           colors=['#4C72B0','#DD8452'], linewidths=2)
ax.scatter(svs[:,0], svs[:,1], s=300, facecolors='none',
           edgecolors='red', lw=3, zorder=6)
# Draw the margin width arrow between the two SVs
ax.annotate('', xy=sv_red, xytext=sv_blue,
            arrowprops=dict(arrowstyle='<->', color='green',
                            lw=2.5, mutation_scale=18))
margin_w = np.linalg.norm(sv_red - sv_blue)
mid_sv = (sv_blue + sv_red)/2
ax.text(mid_sv[0]+0.1, mid_sv[1]+0.15,
        f'Margin = {margin_w:.2f}\n(SVM maximises this!)',
        fontsize=9, color='green', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#e8f5e9', edgecolor='green'))
ax.set_title('Step 4\nMargin = gap between SVs\nSVM maximises this gap!', fontweight='bold')

plt.suptitle('How the Margin is Drawn — Step by Step',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Summary of the 4 steps:')
print('  1. Find the closest blue & red points to each other')
print('  2. Draw a margin line through each (these are the dashed lines)')
print('  3. Place the decision boundary exactly halfway between them')
print('  4. The gap between the two margin lines = the margin. SVM maximises it.')


---
### 📊 The Decision Score: How SVM Scores Every Point

After the boundary is drawn, every point in the space gets a **score**.
This score tells you how far a point is from the boundary — and which side it is on.

```
   score < -1   →  Firmly Blue  (far from boundary, easy case)
   score = -1   →  RIGHT on the Blue margin line  ← Blue support vectors live here
   score =  0   →  Exactly on the decision boundary
   score = +1   →  RIGHT on the Red margin line   ← Red support vectors live here
   score > +1   →  Firmly Red   (far from boundary, easy case)
```

**The support vector rule:**  
A point becomes a support vector if its score is between **-1 and +1** — it is either right on a margin line, or inside the margin zone.

Points with scores beyond ±1 are easy cases — they are far from the boundary and have no influence on it.  
You could delete them and the boundary would not change at all.

> 💡 **Simple rule:**  
> - `|score| > 1` → the point is **outside** the margin → it is **NOT** a support vector → ignored  
> - `|score| ≤ 1` → the point is **inside or on** the margin → it **IS** a support vector → it defines the model

In [ ]:
# ── Decision score visualisation ─────────────────────────────────────────────
np.random.seed(0)
X_sc = np.vstack([np.random.randn(25,2)*0.6+[1,1],
                  np.random.randn(25,2)*0.6+[4,4]])
y_sc = np.array([0]*25 + [1]*25)
clf_sc = SVC(kernel='linear', C=1)
clf_sc.fit(X_sc, y_sc)

xx_sc, yy_sc = np.meshgrid(np.linspace(-1,7,300), np.linspace(-1,7,300))
scores_grid = clf_sc.decision_function(
    np.c_[xx_sc.ravel(), yy_sc.ravel()]).reshape(xx_sc.shape)
scores_pts  = clf_sc.decision_function(X_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: score heatmap ───────────────────────────────────────────────────────
ax = axes[0]
cf = ax.contourf(xx_sc, yy_sc, scores_grid, levels=20, cmap='RdBu_r', alpha=0.85)
plt.colorbar(cf, ax=ax, label='Decision score')
ax.contour(xx_sc, yy_sc, scores_grid, levels=[-1], linestyles='--',
           colors='#4C72B0', linewidths=2.5)
ax.contour(xx_sc, yy_sc, scores_grid, levels=[0],  linestyles='-',
           colors='black', linewidths=3)
ax.contour(xx_sc, yy_sc, scores_grid, levels=[1],  linestyles='--',
           colors='#DD8452', linewidths=2.5)
colors_sc = ['#4C72B0' if c==0 else '#DD8452' for c in y_sc]
ax.scatter(X_sc[:,0], X_sc[:,1], c=colors_sc, s=70, edgecolors='k', zorder=5)
ax.scatter(clf_sc.support_vectors_[:,0], clf_sc.support_vectors_[:,1],
           s=250, facecolors='none', edgecolors='yellow', lw=3, zorder=6,
           label='Support Vectors (score ≈ ±1)')
# Labels on the map
ax.text(0.5, 5.8,'score ≪ 0\n(firmly Blue)',fontsize=9,color='navy',fontweight='bold',
        ha='center',bbox=dict(boxstyle='round',facecolor='#dbeafe',alpha=0.9))
ax.text(5.0, 0.5,'score ≫ 0\n(firmly Red)',fontsize=9,color='darkred',fontweight='bold',
        ha='center',bbox=dict(boxstyle='round',facecolor='#fee2e2',alpha=0.9))
ax.text(2.8, 3.0,'score = 0\n(boundary)',fontsize=9,color='black',fontweight='bold',
        ha='center',bbox=dict(boxstyle='round',facecolor='white',edgecolor='black',alpha=0.9))
ax.set_title('Decision Score Map\n'
             'Support Vectors sit at score = ±1 (the margin lines)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

# ── Right: score distribution bar ────────────────────────────────────────────
ax = axes[1]
sv_mask = np.zeros(len(X_sc), dtype=bool)
sv_mask[clf_sc.support_] = True

blue_sv_scores   = scores_pts[(y_sc==0) &  sv_mask]
blue_nonsv_scores= scores_pts[(y_sc==0) & ~sv_mask]
red_sv_scores    = scores_pts[(y_sc==1) &  sv_mask]
red_nonsv_scores = scores_pts[(y_sc==1) & ~sv_mask]

ax.scatter(blue_nonsv_scores, ['Non-SV (ignored)']*len(blue_nonsv_scores),
           c='#4C72B0', s=70, edgecolors='k', alpha=0.5, label='Blue non-SV')
ax.scatter(blue_sv_scores,    ['Support Vector']*len(blue_sv_scores),
           c='#4C72B0', s=120, edgecolors='red', lw=2.5, alpha=1, label='Blue SV')
ax.scatter(red_nonsv_scores,  ['Non-SV (ignored)']*len(red_nonsv_scores),
           c='#DD8452', s=70, edgecolors='k', alpha=0.5, label='Red non-SV')
ax.scatter(red_sv_scores,     ['Support Vector']*len(red_sv_scores),
           c='#DD8452', s=120, edgecolors='red', lw=2.5, alpha=1, label='Red SV')

ax.axvline(-1, color='#4C72B0', lw=2, linestyle='--', label='Score = -1 (blue margin)')
ax.axvline( 0, color='black',   lw=2, linestyle='-',  label='Score = 0  (boundary)')
ax.axvline( 1, color='#DD8452', lw=2, linestyle='--', label='Score = +1 (red margin)')
ax.axvspan(-1, 1, alpha=0.08, color='gold', label='Margin zone')
ax.set_xlabel('Decision Score', fontsize=11)
ax.set_title('Where do Support Vectors sit on the score scale?\n'
             'SVs are always inside the margin zone (|score| ≤ 1)', fontweight='bold')
ax.legend(fontsize=8, loc='upper left')

plt.suptitle('The Decision Score — SVM\'s Way of Measuring Distance from the Boundary',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Decision scores for support vectors:')
sv_scores_vals = scores_pts[clf_sc.support_]
for i, s in enumerate(sv_scores_vals):
    print(f'  SV {i+1}: score = {s:.3f}  (should be close to ±1)')
print()
print('Non-SV score range:')
nonsv_scores = scores_pts[~sv_mask]
print(f'  min={nonsv_scores.min():.2f}  max={nonsv_scores.max():.2f}  (all outside ±1)')


---
# ⭕ Part 4 — Support Vectors: The "Important" Points

**Support Vectors** are the data points that are **closest to the decision boundary** — they sit right on the margin lines (the dashed lines).

### Why are they called "support" vectors?
Because they **"support"** or **define** the decision boundary. If you removed every other training point and kept only the support vectors, the decision boundary would be **exactly the same**.

### Key insight:
- Only **2–5% of your data** are typically support vectors
- All other points are **irrelevant** to the model
- This makes SVM **very memory-efficient** — it only needs to remember a handful of points

### Intuition:
Think of building a fence between two properties:
- The fence position is decided only by the **boundary stones** at each end
- All the other land in between doesn't affect where the fence goes
- The boundary stones = support vectors

In [ ]:
np.random.seed(42)
n = 30
X_sv = np.vstack([np.random.randn(n,2)*0.7+[1,1],
                  np.random.randn(n,2)*0.7+[4,4]])
y_sv = np.array([0]*n+[1]*n)

clf_sv = SVC(kernel='linear', C=1)
clf_sv.fit(X_sv, y_sv)
sv_idx = clf_sv.support_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (ax, title) in enumerate(zip(axes, ['All Training Points', 'Only Support Vectors — Same Boundary!'])):
    xx, yy = np.meshgrid(np.linspace(-2, 7, 300), np.linspace(-2, 7, 300))
    Z = clf_sv.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    Z2 = clf_sv.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.15, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='black', linewidths=2)
    ax.contour(xx, yy, Z2, levels=[-1, 1], linestyles='--',
               colors=[COLORS[0], COLORS[1]], linewidths=1.5)

    if ax_idx == 0:
        # Show all points
        colors_all = [COLORS[0] if c == 0 else COLORS[1] for c in y_sv]
        ax.scatter(X_sv[:,0], X_sv[:,1], c=colors_all, s=50, edgecolors='k', zorder=3, alpha=0.6,
                   label=f'All {len(X_sv)} training points')
        ax.scatter(clf_sv.support_vectors_[:,0], clf_sv.support_vectors_[:,1],
                   s=200, facecolors='none', edgecolors='red', linewidths=2.5, zorder=5,
                   label=f'Support Vectors ({len(clf_sv.support_vectors_)} pts)')
    else:
        # Show ONLY support vectors
        sv = clf_sv.support_vectors_
        sv_y = y_sv[sv_idx]
        colors_sv = [COLORS[0] if c == 0 else COLORS[1] for c in sv_y]
        ax.scatter(sv[:,0], sv[:,1], c=colors_sv, s=150, edgecolors='k', zorder=5)
        ax.scatter(sv[:,0], sv[:,1], s=300, facecolors='none', edgecolors='red',
                   linewidths=2.5, zorder=6, label=f'Only {len(sv)} support vectors shown')
        ax.text(2.5, -1.5, '🔑 Same boundary!\nRest of the data\ndoesn\'t matter.',
                fontsize=11, color='darkred', fontweight='bold',
                bbox=dict(facecolor='#fff5f5', edgecolor='red', boxstyle='round,pad=0.4'))

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(-2, 7); ax.set_ylim(-2, 7)

plt.suptitle('Support Vectors Define the Entire Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

pct = len(clf_sv.support_vectors_)/len(X_sv)*100
print(f"Total training points   : {len(X_sv)}")
print(f"Support Vectors         : {len(clf_sv.support_vectors_)} ({pct:.1f}% of data)")
print(f"\n🎯 Only {pct:.1f}% of the data actually DEFINES the model. The rest could be deleted!")

---
# 🧹 Part 5 — Hard Margin vs Soft Margin

### Hard Margin SVM

The **"strict" version** of SVM. It says:
> *"Every single point MUST be on the correct side of the boundary. Zero errors allowed."*

**Problem:** Real data is messy. There are:
- Outliers
- Overlapping classes  
- Measurement errors

A hard margin SVM **fails completely** when data isn't perfectly separable.

### Soft Margin SVM

The **"flexible" version** of SVM. It says:
> *"I'll allow SOME points to be on the wrong side or inside the margin, if it leads to a better overall boundary."*

This is what sklearn uses by default, and it's controlled by the **C parameter** (explained next).

In [ ]:
np.random.seed(10)
n = 30
X_hard = np.vstack([
    np.random.randn(n, 2)*0.7 + [1, 1],
    np.random.randn(n, 2)*0.7 + [3.5, 3.5],
    [[2.5, 1.8]]  # One outlier in wrong class area
])
y_hard = np.array([0]*n + [1]*n + [0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def plot_boundary(clf, X, y, ax, title):
    xx, yy = np.meshgrid(np.linspace(-1,6,300), np.linspace(-1,6,300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    Z2 = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='k', linewidths=2)
    ax.contour(xx, yy, Z2, levels=[-1,1], linestyles='--',
               colors=[COLORS[0], COLORS[1]], linewidths=1.5)
    colors_pts = [COLORS[0] if c==0 else COLORS[1] for c in y]
    ax.scatter(X[:,0], X[:,1], c=colors_pts, s=60, edgecolors='k', zorder=3)
    ax.scatter(clf.support_vectors_[:,0], clf.support_vectors_[:,1],
               s=200, facecolors='none', edgecolors='red', lw=2.5, zorder=5)
    # Highlight the outlier
    ax.scatter([2.5], [1.8], c='lime', s=200, edgecolors='black', lw=2,
               zorder=6, marker='*', label='Outlier ⚠️')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(-1,6); ax.set_ylim(-1,6)

# Hard margin (very large C)
clf_hard = SVC(kernel='linear', C=10000)
clf_hard.fit(X_hard, y_hard)
plot_boundary(clf_hard, X_hard, y_hard, axes[0],
              '❌ Hard Margin (C=10000)\nDistorted by outlier!')

# Soft margin (reasonable C)
clf_soft = SVC(kernel='linear', C=1)
clf_soft.fit(X_hard, y_hard)
plot_boundary(clf_soft, X_hard, y_hard, axes[1],
              '✅ Soft Margin (C=1)\nIgnores outlier, better boundary!')

plt.suptitle('Hard Margin vs Soft Margin SVM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 The outlier (⭐) completely warps the hard margin boundary.")
print("   Soft margin handles it gracefully — this is why we prefer soft margin in practice.")

---
# 🎛️ Part 6 — The C Parameter (Explained from Scratch)

**C stands for "Cost" or "Constraint".**

It controls the **trade-off** between:
- Making the **margin as wide as possible** (good generalization)
- **Correctly classifying** all training points (avoiding errors)

These two goals often **conflict** with each other.

### Think of it like a teacher marking exams:

| Teacher Style | C equivalent | Behavior |
|---------------|-------------|----------|
| **Very strict** — fails you for one wrong answer | Large C | Forces correct classification → narrow margin → risk of overfitting |
| **Reasonable** — some flexibility | Medium C (1–10) | Balanced: some errors allowed, decent margin |
| **Very lenient** — passes everyone | Small C | Allows many errors → very wide margin → risk of underfitting |

### Overfitting vs Underfitting:
- **Overfitting** (large C): Model memorizes training data too well. Works great on training data, fails on new data.
- **Underfitting** (small C): Model is too simple. Doesn't even learn the training data well.
- **Just right**: We use cross-validation to find the best C.

In [ ]:
np.random.seed(5)
X_c, y_c = make_classification(n_samples=120, n_features=2, n_redundant=0,
                                 n_clusters_per_class=1, class_sep=0.9, random_state=5)
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(X_c, y_c, test_size=0.3, random_state=42)

C_values  = [0.001, 0.1, 1, 10, 1000]
train_acc = []
test_acc  = []
sv_counts = []

for C in C_values:
    m = SVC(kernel='rbf', C=C, gamma='scale')
    m.fit(X_c_train, y_c_train)
    train_acc.append(accuracy_score(y_c_train, m.predict(X_c_train)))
    test_acc.append(accuracy_score(y_c_test, m.predict(X_c_test)))
    sv_counts.append(len(m.support_vectors_))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Accuracy vs C
ax = axes[0]
ax.semilogx(C_values, train_acc, 'o-', color=COLORS[0], lw=2, ms=8, label='Train Accuracy')
ax.semilogx(C_values, test_acc,  's-', color=COLORS[1], lw=2, ms=8, label='Test Accuracy')
ax.axvline(1, color='green', lw=1.5, linestyle=':', label='C=1 (good start)')
ax.set_xlabel('C value (log scale)', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy vs C\n(Train vs Test)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0.5, 1.05)

# Support vectors vs C
ax = axes[1]
ax.semilogx(C_values, sv_counts, 'D-', color='purple', lw=2, ms=8)
ax.set_xlabel('C value (log scale)', fontsize=11)
ax.set_ylabel('Number of Support Vectors', fontsize=11)
ax.set_title('Support Vectors vs C\n(High C → fewer SVs)', fontweight='bold')
for x, y in zip(C_values, sv_counts):
    ax.annotate(str(y), (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

# Decision boundary comparison for extreme C values
ax = axes[2]
C_demo_vals = [0.01, 1, 1000]
labels = ['C=0.01\n(Underfitting)', 'C=1\n(Balanced ✅)', 'C=1000\n(Overfitting risk)']
bar_colors = ['#e07b54', '#4CAF50', '#e74c3c']
bars = ax.bar(labels, [test_acc[0], test_acc[2], test_acc[4]],
              color=bar_colors, edgecolor='k', width=0.5)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.2%}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Test Accuracy', fontsize=11)
ax.set_title('Test Accuracy for Different C', fontweight='bold')
ax.set_ylim(0.5, 1.05)

plt.suptitle('Effect of C Parameter on SVM Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🎯 Key takeaways:")
print("   - Very small C  → Low accuracy (underfitting)")
print("   - Very large C  → High train acc but may overfit")
print("   - C=1 to 10     → Usually a good starting range")
print("   - Large C       → fewer support vectors (very strict boundary)")

---
# 🪄 Part 7 — The Kernel Trick (SVM's Superpower)

### The Problem

What if your data **cannot** be separated by a straight line? Like this:

In [ ]:
X_circ, y_circ = make_circles(n_samples=200, noise=0.08, factor=0.4, random_state=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 2D view (can't separate)
ax = axes[0]
colors_c = [COLORS[0] if c==0 else COLORS[1] for c in y_circ]
ax.scatter(X_circ[:,0], X_circ[:,1], c=colors_c, s=60, edgecolors='k')
ax.set_title('❌ 2D Data — No straight line can\nseparate inner vs outer circle!',
             fontweight='bold')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

# 3D view after kernel transform
from mpl_toolkits.mplot3d import Axes3D
ax3d = fig.add_subplot(122, projection='3d')
# Add 3rd dimension: z = x^2 + y^2  (this is what RBF effectively does)
z_lift = X_circ[:,0]**2 + X_circ[:,1]**2
ax3d.scatter(X_circ[:,0], X_circ[:,1], z_lift, c=colors_c, s=40, edgecolors='none', alpha=0.8)

# Draw the separating plane in 3D
xx3, yy3 = np.meshgrid(np.linspace(-1.5,1.5,20), np.linspace(-1.5,1.5,20))
z_plane = np.full_like(xx3, 0.21)
ax3d.plot_surface(xx3, yy3, z_plane, alpha=0.3, color='green')
ax3d.set_title('✅ Lifted to 3D — A flat\nplane can separate them!',
               fontweight='bold')
ax3d.set_xlabel('Feature 1'); ax3d.set_ylabel('Feature 2'); ax3d.set_zlabel('z = x²+y²')

plt.suptitle('The Kernel Trick: Lifting Data to Higher Dimensions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 The Kernel Trick:")
print("   1. Data is NOT linearly separable in 2D")
print("   2. We ADD a new dimension (z = x²+y²)")
print("   3. Now a flat plane CAN separate the data in 3D")
print("   4. Kernel functions do this WITHOUT actually computing the transformation!")
print("      (That's the 'trick' — it's computationally much cheaper)")

---
# 🔷 Part 8 — Linear Kernel

### What it does
The simplest kernel. It draws a **straight-line** (or hyperplane in higher dimensions) boundary.

No transformation is applied — the data is used as-is.

### Mathematical Formula

$$K(x_i, x_j) = x_i \cdot x_j$$

In plain English: The kernel between two points is just their **dot product** (a measure of how similar they are, computed as the sum of products of corresponding features).

For example, if point A = (2, 3) and point B = (1, 4):
$$K(A, B) = (2 \times 1) + (3 \times 4) = 2 + 12 = 14$$

### When to use it:
- Data is (roughly) linearly separable
- You have **many features** (text data, gene expression)
- You want **fast training**
- Interpretability matters

### Parameters:
Only **C** (no gamma, no degree needed).

In [ ]:
# Test on 3 different datasets
np.random.seed(42)
datasets_lin = [
    (make_classification(n_samples=150, n_features=2, n_redundant=0,
                         n_clusters_per_class=1, class_sep=2.5, random_state=0), 'Well-separated (✅ Linear works great)'),
    (make_classification(n_samples=150, n_features=2, n_redundant=0,
                         n_clusters_per_class=1, class_sep=0.7, random_state=0), 'Overlapping (⚠️ Linear struggles)'),
    (make_circles(n_samples=150, noise=0.1, factor=0.4, random_state=0),         'Circles (❌ Linear fails)'),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))

for col, ((X_d, y_d), title) in enumerate(datasets_lin):
    sc = StandardScaler()
    X_s = sc.fit_transform(X_d)
    clf = SVC(kernel='linear', C=1)
    clf.fit(X_s, y_d)
    acc = accuracy_score(y_d, clf.predict(X_s))

    # Top row: raw data
    ax = axes[0][col]
    colors_d = [COLORS[0] if c==0 else COLORS[1] for c in y_d]
    ax.scatter(X_s[:,0], X_s[:,1], c=colors_d, s=55, edgecolors='k', alpha=0.8)
    ax.set_title(f'Data: {title}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

    # Bottom row: boundary
    ax = axes[1][col]
    xx, yy = np.meshgrid(np.linspace(X_s[:,0].min()-0.5, X_s[:,0].max()+0.5, 300),
                         np.linspace(X_s[:,1].min()-0.5, X_s[:,1].max()+0.5, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    Z2 = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='k', linewidths=2)
    ax.contour(xx, yy, Z2, levels=[-1,1], linestyles='--',
               colors=[COLORS[0], COLORS[1]], linewidths=1.5)
    ax.scatter(X_s[:,0], X_s[:,1], c=colors_d, s=40, edgecolors='k', alpha=0.6, zorder=3)
    ax.scatter(clf.support_vectors_[:,0], clf.support_vectors_[:,1],
               s=180, facecolors='none', edgecolors='red', lw=2.5, zorder=5)
    ax.set_title(f'Linear Kernel Result\nAccuracy: {acc:.1%}', fontsize=10, fontweight='bold')

plt.suptitle('Linear Kernel — When It Works & When It Fails', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 Linear Kernel Summary:")
print("   Formula : K(xᵢ, xⱼ) = xᵢ · xⱼ   (simple dot product)")
print("   Best for: High-dimensional data, text, genes, linearly-separable data")
print("   Fails on: Non-linear patterns (circles, spirals, moons)")
print("   Params  : Only C")
print("   Speed   : Fastest kernel")

---
# 🔴 Part 9 — RBF / Gaussian Kernel

### What it does
**RBF = Radial Basis Function** (also called Gaussian Kernel)

This is the **most popular and powerful** kernel. It can draw **any curved boundary** — circles, blobs, complex shapes.

It works by measuring how **similar** two points are based on their **distance**:
- Points that are **close** to each other → high similarity → large kernel value
- Points that are **far** from each other → low similarity → small kernel value

### Mathematical Formula

$$K(x_i, x_j) = \exp\left(-\gamma \, ||x_i - x_j||^2\right)$$

Breaking this down:
- $||x_i - x_j||^2$ → the **squared distance** between two points
- $\exp(...)$ → the exponential function (makes values between 0 and 1)
- $\gamma$ → controls how **fast** similarity falls off with distance (explained below)

When distance is **0** (same point): $K = e^0 = 1$ (maximum similarity)  
When distance is **large**: $K \approx 0$ (no similarity)

### The Gamma (γ) Parameter — Explained from Scratch

**Gamma controls how far a single training point's "influence" reaches.**

Imagine each training point as a **radio tower** broadcasting a signal:

| Gamma | Analogy | Effect |
|-------|---------|--------|
| **Large γ** | Weak signal, short range | Each point only influences nearby area → very wiggly boundary → overfitting |
| **Small γ** | Strong signal, long range | Each point influences large area → smooth boundary → may underfit |

> 💡 **Default:** `gamma='scale'` in sklearn = 1/(n_features × variance). Always start here.

In [ ]:
# ── Visualize the Gaussian function itself ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
distances = np.linspace(0, 5, 200)
for gamma, color, lbl in [(0.1,'#2196F3','γ=0.1 (long reach)'),
                           (1,   '#4CAF50','γ=1   (medium)'),
                           (5,   '#F44336','γ=5   (short reach)'),
                           (20,  '#9C27B0','γ=20  (very short)')]:
    K = np.exp(-gamma * distances**2)
    ax.plot(distances, K, color=color, lw=2.5, label=lbl)

ax.set_xlabel('Distance between two points', fontsize=11)
ax.set_ylabel('K(xᵢ, xⱼ) = similarity', fontsize=11)
ax.set_title('RBF Kernel: Similarity Decays with Distance\n(Higher γ = faster decay = shorter reach)',
             fontweight='bold')
ax.legend(fontsize=10)
ax.axvline(1, color='gray', lw=1, linestyle=':')
ax.text(1.05, 0.7, 'distance = 1', color='gray', fontsize=9)

# ── Effect of gamma on decision boundary ─────────────────────────────────────
ax = axes[1]
X_g, y_g = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)

gamma_vals  = [0.01, 0.1, 1, 10, 100]
test_accs_g = []
for gv in gamma_vals:
    m = SVC(kernel='rbf', C=1, gamma=gv)
    m.fit(X_g, y_g)
    test_accs_g.append(accuracy_score(y_g, m.predict(X_g)))

bar_c = ['#F44336' if i==0 else ('#4CAF50' if i in [2,3] else '#FF9800') for i in range(len(gamma_vals))]
bars = ax.bar([str(g) for g in gamma_vals], test_accs_g, color=bar_c, edgecolor='k', width=0.6)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.2%}', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Gamma (γ) value', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy vs Gamma (Circles Dataset)\n(Too small or too large → worse!)', fontweight='bold')
ax.set_ylim(0.4, 1.05)
ax.axhline(max(test_accs_g), color='green', lw=1, linestyle=':', label='Best accuracy')
ax.legend(fontsize=9)

plt.suptitle('RBF Kernel — Gamma Parameter Deep Dive', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual comparison of gamma on decision boundaries ────────────────────────
X_rbf, y_rbf = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)
gamma_show = [0.05, 0.5, 2, 20]
labels_g   = ['γ=0.05\nUnderfit: too smooth',
               'γ=0.5\nGood ✅',
               'γ=2\nGood ✅',
               'γ=20\nOverfit: too wiggly']

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, gv, lbl in zip(axes, gamma_show, labels_g):
    clf = SVC(kernel='rbf', C=1, gamma=gv)
    clf.fit(X_rbf, y_rbf)
    acc = accuracy_score(y_rbf, clf.predict(X_rbf))

    xx, yy = np.meshgrid(np.linspace(-1.6,1.6,300), np.linspace(-1.6,1.6,300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='k', linewidths=2)
    colors_r = [COLORS[0] if c==0 else COLORS[1] for c in y_rbf]
    ax.scatter(X_rbf[:,0], X_rbf[:,1], c=colors_r, s=40, edgecolors='k', alpha=0.7, zorder=3)
    ax.set_title(f'{lbl}\nAcc={acc:.1%} | SVs={len(clf.support_vectors_)}',
                 fontsize=9, fontweight='bold')
    ax.set_xlim(-1.6,1.6); ax.set_ylim(-1.6,1.6)

plt.suptitle('RBF Kernel — Effect of Gamma on Decision Boundary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 RBF Kernel Summary:")
print("   Formula: K(xᵢ,xⱼ) = exp(-γ||xᵢ-xⱼ||²)")
print("   Best for: General purpose — works for most problems!")
print("   Default : gamma='scale' (start here, then tune)")
print("   Params  : C + gamma")
print("   ⚠️ Must scale data first (StandardScaler)")

---
# 🔺 Part 10 — Polynomial Kernel

### What it does
The Polynomial kernel creates **curved, polynomial-shaped** decision boundaries. Think of it as drawing curves like parabolas, hyperbolas, and higher-order curves.

### Mathematical Formula

$$K(x_i, x_j) = (\gamma \cdot x_i \cdot x_j + r)^d$$

Breaking this down:
- $x_i \cdot x_j$ → dot product (same as linear kernel)
- $\gamma$ → scaling factor (controls the weight of high-degree vs low-degree terms)
- $r$ → free parameter (`coef0` in sklearn), shifts the polynomial
- $d$ → **degree** — this is the most important parameter!
  - `d=2` → parabolic boundary
  - `d=3` → cubic boundary
  - `d=1` → same as linear!

### Parameters:
- **C** — regularization (same as always)
- **degree (d)** — polynomial degree (usually 2 or 3)
- **coef0 (r)** — offset term, default=0; increase it if you want the polynomial to "slide"
- **gamma** — scaling factor, default='scale'

### When to use:
- Natural language processing (NLP)
- Image recognition
- Data where you expect polynomial relationships

In [ ]:
# ── Show polynomial kernel formula intuitively ────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

X_poly, y_poly = make_moons(n_samples=200, noise=0.15, random_state=42)
sc_p = StandardScaler()
X_poly_s = sc_p.fit_transform(X_poly)

degrees = [1, 2, 3, 5]
degree_labels = [
    'degree=1\n(same as linear)',
    'degree=2\n(quadratic) ✅',
    'degree=3\n(cubic) ✅',
    'degree=5\n(may overfit)',
]

for ax, deg, lbl in zip(axes, degrees, degree_labels):
    clf = SVC(kernel='poly', degree=deg, C=1, coef0=1, gamma='scale')
    clf.fit(X_poly_s, y_poly)
    acc = accuracy_score(y_poly, clf.predict(X_poly_s))

    xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 300), np.linspace(-2, 2.5, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='k', linewidths=2)
    colors_p = [COLORS[0] if c==0 else COLORS[1] for c in y_poly]
    ax.scatter(X_poly_s[:,0], X_poly_s[:,1], c=colors_p, s=40, edgecolors='k', alpha=0.7, zorder=3)
    ax.scatter(clf.support_vectors_[:,0], clf.support_vectors_[:,1],
               s=150, facecolors='none', edgecolors='red', lw=2, zorder=5)
    ax.set_title(f'{lbl}\nAcc={acc:.1%} | SVs={len(clf.support_vectors_)}',
                 fontsize=9, fontweight='bold')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2, 2.5)

plt.suptitle('Polynomial Kernel — Effect of Degree (Moon Dataset)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 Polynomial Kernel Summary:")
print("   Formula: K(xᵢ,xⱼ) = (γ·xᵢ·xⱼ + r)^d")
print("   Key param: degree d (start with 2 or 3)")
print("   Best for: NLP, image classification, polynomial relationships")
print("   ⚠️  Higher degree → more complex → more risk of overfitting")
print("   ⚠️  Slower to train than linear, faster than RBF on some datasets")

---
# 🔁 Part 11 — Sigmoid Kernel

### What it does
The Sigmoid kernel makes SVM behave similarly to a **neural network** (a 2-layer neural network, to be precise).

It uses the same sigmoid/tanh function used in neural networks.

### Mathematical Formula

$$K(x_i, x_j) = \tanh(\gamma \cdot x_i \cdot x_j + r)$$

Breaking this down:
- $\tanh$ → hyperbolic tangent function (S-shaped curve, outputs between -1 and 1)
- $\gamma$ → scaling factor
- $r$ → offset (`coef0`)

### Important note:
Unlike linear and RBF, sigmoid is **not always a valid kernel** (it doesn't always satisfy the math requirements). It still works in practice for some problems but is rarely the best choice.

### When to use:
- Rarely recommended — try linear/RBF/poly first
- Can work on some neural-network-style problems
- Historical interest: it bridges SVM and neural networks

In [ ]:
# Show the tanh function shape + compare sigmoid vs other kernels on one dataset
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Left: tanh function
ax = axes[0]
z = np.linspace(-4, 4, 200)
for gamma, color, lbl in [(0.5, '#2196F3','γ=0.5'), (1, '#4CAF50','γ=1'), (2, '#F44336','γ=2')]:
    ax.plot(z, np.tanh(gamma*z), color=color, lw=2.5, label=lbl)
ax.axhline(0, color='k', lw=0.8, linestyle='-')
ax.axvline(0, color='k', lw=0.8, linestyle='-')
ax.set_xlabel('γ · xᵢ · xⱼ + r', fontsize=11)
ax.set_ylabel('K(xᵢ, xⱼ) = tanh(...)', fontsize=11)
ax.set_title('Sigmoid Kernel: tanh Function\n(S-shaped, bounded between -1 and 1)', fontweight='bold')
ax.legend()

# Right: Compare all kernels on moons
X_sig, y_sig = make_moons(n_samples=200, noise=0.15, random_state=42)
sc_sig = StandardScaler()
X_sig_s = sc_sig.fit_transform(X_sig)

kernels_all = [('rbf', {}, 'RBF'), ('poly', {'degree':3, 'coef0':1}, 'Poly (d=3)'), ('sigmoid', {'coef0':0}, 'Sigmoid')]
for ax, (kern, kwargs, kname) in zip(axes[1:], kernels_all[1:]):
    clf = SVC(kernel=kern, C=1, gamma='scale', **kwargs)
    clf.fit(X_sig_s, y_sig)
    acc = accuracy_score(y_sig, clf.predict(X_sig_s))
    xx, yy = np.meshgrid(np.linspace(-2.5,2.5,300), np.linspace(-2,2.5,300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='k', linewidths=2)
    colors_s = [COLORS[0] if c==0 else COLORS[1] for c in y_sig]
    ax.scatter(X_sig_s[:,0], X_sig_s[:,1], c=colors_s, s=40, edgecolors='k', alpha=0.7, zorder=3)
    ax.set_title(f'{kname} Kernel\nAcc={acc:.1%}', fontsize=11, fontweight='bold')
    ax.set_xlim(-2.5,2.5); ax.set_ylim(-2,2.5)

plt.suptitle('Sigmoid Kernel vs Others', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 Sigmoid Kernel Summary:")
print("   Formula: K(xᵢ,xⱼ) = tanh(γ·xᵢ·xⱼ + r)")
print("   Best for: Rarely recommended; use RBF first")
print("   Interesting fact: Equivalent to a simple 2-layer neural network!")
print("   ⚠️  Not always a valid kernel — can produce unstable results")

---
# 📊 Part 12 — Kernel Comparison: All at Once

Let's do a comprehensive visual comparison of all kernels across multiple datasets.

In [ ]:
np.random.seed(42)
all_datasets = [
    (make_classification(n_samples=150, n_features=2, n_redundant=0,
                         n_clusters_per_class=1, class_sep=2, random_state=1), 'Blobs'),
    (make_circles(n_samples=150, noise=0.08, factor=0.4, random_state=1),      'Circles'),
    (make_moons(n_samples=150, noise=0.1, random_state=1),                     'Moons'),
]
all_kernels = [
    ('linear',  {},                                   'Linear'),
    ('rbf',     {'gamma': 'scale'},                   'RBF'),
    ('poly',    {'degree': 3, 'coef0': 1},            'Poly (d=3)'),
    ('sigmoid', {'coef0': 0},                         'Sigmoid'),
]

fig, axes = plt.subplots(len(all_datasets), len(all_kernels), figsize=(20, 13))
row_labels = [d[1] for d in all_datasets]
col_labels = [k[2] for k in all_kernels]

for row, ((X_d, y_d), ds_name) in enumerate(all_datasets):
    sc = StandardScaler()
    X_d_s = sc.fit_transform(X_d)
    X_tr, X_te, y_tr, y_te = train_test_split(X_d_s, y_d, test_size=0.3, random_state=42)

    for col, (kern, kw, kname) in enumerate(all_kernels):
        ax = axes[row][col]
        clf = SVC(kernel=kern, C=1, **kw)
        clf.fit(X_tr, y_tr)
        tr_acc = accuracy_score(y_tr, clf.predict(X_tr))
        te_acc = accuracy_score(y_te, clf.predict(X_te))

        xx, yy = np.meshgrid(np.linspace(X_d_s[:,0].min()-0.5, X_d_s[:,0].max()+0.5, 250),
                             np.linspace(X_d_s[:,1].min()-0.5, X_d_s[:,1].max()+0.5, 250))
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
        ax.contour(xx, yy, Z, colors='k', linewidths=1.5)
        colors_all = [COLORS[0] if c==0 else COLORS[1] for c in y_d]
        ax.scatter(X_d_s[:,0], X_d_s[:,1], c=colors_all, s=25, edgecolors='k', alpha=0.6, zorder=3, lw=0.5)

        if row == 0:
            ax.set_title(f'Kernel: {kname}', fontsize=11, fontweight='bold', pad=10)
        if col == 0:
            ax.set_ylabel(f'Dataset:\n{ds_name}', fontsize=10, fontweight='bold')

        color_box = '#d4edda' if te_acc >= 0.85 else ('#fff3cd' if te_acc >= 0.70 else '#f8d7da')
        ax.text(0.03, 0.97, f'Test: {te_acc:.0%}',
                transform=ax.transAxes, fontsize=9, va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color_box, edgecolor='gray', alpha=0.9))

plt.suptitle('All 4 Kernels × 3 Datasets — Comprehensive Comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("🟢 Green box = Test Acc ≥ 85%  |  🟡 Yellow = 70–85%  |  🔴 Red = < 70%")
print()
print("Key observations:")
print("  • Linear works great on blobs, fails badly on circles/moons")
print("  • RBF handles all three datasets well")
print("  • Poly (d=3) is good on moons, mixed on others")
print("  • Sigmoid is inconsistent — not recommended as first choice")

---
# 🏥 Part 13 — Full Real-World Example: Breast Cancer Dataset

Let's apply SVM to a real medical dataset: **Breast Cancer Wisconsin**.

### The Task:
Given measurements from a breast mass biopsy (like radius, texture, perimeter, smoothness...), predict whether the tumor is **Malignant (cancerous)** or **Benign (non-cancerous)**.

### Why SVM is great here:
- Medical data often has many features (30 here)
- We need reliable predictions, not just high accuracy
- Margins help us understand confidence of predictions

### Step-by-Step Workflow:

In [ ]:
# ── STEP 1: Load and explore the data ────────────────────────────────────────
cancer = datasets.load_breast_cancer()
X_bc = cancer.data
y_bc = cancer.target     # 0 = malignant, 1 = benign

df = pd.DataFrame(X_bc, columns=cancer.feature_names)
df['target'] = y_bc
df['diagnosis'] = ['Benign' if t else 'Malignant' for t in y_bc]

print("📋 Dataset Overview")
print(f"   Samples  : {X_bc.shape[0]}")
print(f"   Features : {X_bc.shape[1]}")
print(f"   Classes  : {cancer.target_names.tolist()}")
print()
print("Class distribution:")
print(df['diagnosis'].value_counts().to_string())
print()
print("First 5 feature names:")
print("  ", cancer.feature_names[:5].tolist())

In [ ]:
# ── STEP 2: Visualize the data ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Class distribution pie
ax = axes[0]
counts = df['diagnosis'].value_counts()
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=['#DD8452', '#4C72B0'], startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Class Distribution', fontweight='bold')

# Feature distributions for a key feature
ax = axes[1]
benign    = df[df['target']==1]['mean radius']
malignant = df[df['target']==0]['mean radius']
ax.hist(benign,    bins=25, alpha=0.7, color=COLORS[0], label='Benign',    edgecolor='k')
ax.hist(malignant, bins=25, alpha=0.7, color=COLORS[1], label='Malignant', edgecolor='k')
ax.set_xlabel('Mean Radius', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Feature Distribution: Mean Radius\n(Malignant tumors tend to be larger)', fontweight='bold')
ax.legend()

# Scatter plot: 2 features
ax = axes[2]
colors_bc = [COLORS[0] if t==1 else COLORS[1] for t in y_bc]
ax.scatter(X_bc[:, 0], X_bc[:, 2], c=colors_bc, s=30, alpha=0.6, edgecolors='none')
patch_b = mpatches.Patch(color=COLORS[0], label='Benign')
patch_m = mpatches.Patch(color=COLORS[1], label='Malignant')
ax.legend(handles=[patch_b, patch_m])
ax.set_xlabel('Mean Radius', fontsize=11)
ax.set_ylabel('Mean Perimeter', fontsize=11)
ax.set_title('Benign vs Malignant\n(2 features)', fontweight='bold')

plt.suptitle('Breast Cancer Dataset — Exploratory Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── STEP 3: Split the data ─────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_bc, y_bc, test_size=0.20, random_state=42, stratify=y_bc
)
print(f"Training set : {len(X_tr)} samples")
print(f"Test set     : {len(X_te)} samples")
print()

# ── STEP 4: Scale features ─────────────────────────────────────────────────────
# WHY SCALE? SVM is sensitive to feature magnitudes.
# 'mean radius' ranges 6–28, 'mean fractal dimension' ranges 0.05–0.09
# Without scaling, large-valued features dominate. Scaling fixes this.

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)   # Learn mean/std FROM TRAINING DATA ONLY
X_te_s = scaler.transform(X_te)       # Apply same transform to test data

print("Before scaling — Feature ranges:")
print(f"  Mean radius     : {X_tr[:,0].min():.1f} – {X_tr[:,0].max():.1f}")
print(f"  Mean smoothness : {X_tr[:,4].min():.4f} – {X_tr[:,4].max():.4f}")
print()
print("After scaling — Feature ranges:")
print(f"  Mean radius     : {X_tr_s[:,0].min():.2f} – {X_tr_s[:,0].max():.2f}")
print(f"  Mean smoothness : {X_tr_s[:,4].min():.2f} – {X_tr_s[:,4].max():.2f}")
print()
print("✅ Now both features are on the same scale — SVM can treat them fairly!")

In [ ]:
# ── STEP 5: Train and evaluate ────────────────────────────────────────────────
svm_bc = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
svm_bc.fit(X_tr_s, y_tr)

y_pred_bc = svm_bc.predict(X_te_s)

print("=" * 50)
print(f"  Test Accuracy  : {accuracy_score(y_te, y_pred_bc):.2%}")
print("=" * 50)
print()
print("📊 Classification Report:")
print(classification_report(y_te, y_pred_bc, target_names=['Malignant', 'Benign']))

# Explain the report
print("📖 Reading the report:")
print("   Precision = Of all predicted positives, how many were actually positive?")
print("   Recall    = Of all actual positives, how many did we catch?")
print("   F1-Score  = Harmonic mean of precision & recall (balanced metric)")
print("   Support   = Number of actual instances of each class")

In [ ]:
# ── STEP 6: Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_te, y_pred_bc)

fig, ax = plt.subplots(figsize=(7, 5.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Malignant', 'Benign'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — SVM on Breast Cancer', fontsize=13, fontweight='bold')

# Add annotations
ax.text(0, 0, f'\nTrue Negative\n(Correctly\nidentified\nMalignant)', ha='center', va='top',
        fontsize=8, color='white', fontweight='bold')
ax.text(1, 1, f'\nTrue Positive\n(Correctly\nidentified\nBenign)', ha='center', va='top',
        fontsize=8, color='darkblue', fontweight='bold')

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (Malignant correctly identified): {tn}")
print(f"True Positives  (Benign correctly identified)   : {tp}")
print(f"False Positives (Benign said to be Malignant)   : {fp}  ← Over-cautious")
print(f"False Negatives (Malignant said to be Benign)   : {fn}  ← Dangerous in medicine!")
print()
print("⚕️  In medical diagnosis, False Negatives are the most dangerous")
print("   (missing a real cancer). We'd want high Recall for the malignant class.")

---
# 🔧 Part 14 — Hyperparameter Tuning: Finding the Best C and Gamma

### What is Hyperparameter Tuning?

**Parameters** are things the model learns automatically from data (like the weights/coefficients).

**Hyperparameters** are settings **you** choose before training. For SVM:
- `C` — regularization strength
- `gamma` — influence radius (for RBF)
- `kernel` — which kernel to use

Choosing them by hand is guesswork. We use **GridSearchCV** to automate this.

### What is Cross-Validation?

Instead of just one train/test split, cross-validation does **k splits**:
1. Split data into 5 equal parts (folds)
2. Train on 4 folds, test on the remaining 1
3. Repeat 5 times (each fold gets to be the test set once)
4. Average the 5 accuracy scores

This gives a much more **reliable estimate** of real performance.

In [ ]:
# ── Visualize cross-validation ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
n_folds = 5
n_total = 10

for fold in range(n_folds):
    for i in range(n_total):
        if i == fold * 2 or i == fold * 2 + 1:  # approximate
            color = '#DD8452'  # test
            label = 'Test' if (fold == 0 and i == 0) else ''
        else:
            color = '#4C72B0'  # train
            label = 'Train' if (fold == 0 and i == 2) else ''
        rect = plt.Rectangle([i, n_folds-fold-1], 0.9, 0.9,
                              facecolor=color, edgecolor='white', linewidth=2)
        ax.add_patch(rect)
    ax.text(-0.5, n_folds-fold-0.55, f'Fold {fold+1}', va='center', ha='right',
            fontsize=10, fontweight='bold')

ax.set_xlim(-1, n_total+0.5)
ax.set_ylim(-0.2, n_folds+0.2)
ax.axis('off')
blue_patch = mpatches.Patch(color='#4C72B0', label='Training data')
orange_patch = mpatches.Patch(color='#DD8452', label='Validation (test) data')
ax.legend(handles=[blue_patch, orange_patch], loc='upper right', fontsize=11)
ax.set_title('5-Fold Cross Validation — Each fold takes a turn as the test set',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Run GridSearchCV ──────────────────────────────────────────────────────────
param_grid = {
    'C'    : [0.01, 0.1, 1, 10, 100, 1000],
    'gamma': [10, 1, 0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf']
}

print(f"Testing {len(param_grid['C'])} × {len(param_grid['gamma'])} = "
      f"{len(param_grid['C'])*len(param_grid['gamma'])} parameter combinations")
print("Each tested with 5-fold CV = "
      f"{len(param_grid['C'])*len(param_grid['gamma'])*5} model fits total")
print("Running...")

grid_search = GridSearchCV(
    SVC(), param_grid,
    cv=5, scoring='accuracy',
    n_jobs=-1, verbose=0
)
grid_search.fit(X_tr_s, y_tr)

print(f"\n🏆 Best Parameters : {grid_search.best_params_}")
print(f"📈 Best CV Accuracy: {grid_search.best_score_:.4%}")
best = grid_search.best_estimator_
y_pred_best = best.predict(X_te_s)
print(f"🎯 Test Accuracy   : {accuracy_score(y_te, y_pred_best):.4%}")

In [ ]:
# ── Visualize the search results as a heatmap ─────────────────────────────────
results_df = pd.DataFrame(grid_search.cv_results_)
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_gamma',
    columns='param_C'
)

fig, ax = plt.subplots(figsize=(10, 6))
hm = sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                 ax=ax, linewidths=0.5, vmin=0.85, vmax=1.0,
                 annot_kws={'size': 9})
ax.set_title('GridSearchCV Heatmap — Cross-Validation Accuracy\n'
             '(Brighter Green = Better)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('C (regularization)', fontsize=12)
ax.set_ylabel('Gamma', fontsize=12)

# Mark best
best_C     = grid_search.best_params_['C']
best_gamma = grid_search.best_params_['gamma']
C_pos     = list(pivot.columns).index(best_C)
gamma_pos = list(pivot.index).index(best_gamma)
ax.add_patch(plt.Rectangle([C_pos, gamma_pos], 1, 1,
                            fill=False, edgecolor='blue', lw=4, zorder=5))
ax.text(C_pos+0.5, gamma_pos-0.2, '⭐ Best', ha='center',
        fontsize=10, color='blue', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"⭐ Best combination: C={best_C}, gamma={best_gamma}")
print("💡 The blue box shows where the best combination is.")
print("   Notice that both extremes (top-left AND bottom-right) perform poorly.")

---
# 🌐 Part 15 — SVM for Multi-Class Problems

SVM is natively a **binary classifier** (two classes only). But what if you have 3 or more classes?

Sklearn handles this automatically using two strategies:

### Strategy 1: One-vs-Rest (OVR)
For each class, train one SVM:
- SVM 1: "Is it Class A vs everything else?"
- SVM 2: "Is it Class B vs everything else?"
- SVM 3: "Is it Class C vs everything else?"
→ Pick the class with the highest confidence score

### Strategy 2: One-vs-One (OVO)
For every PAIR of classes, train one SVM:
- SVM 1: Class A vs Class B
- SVM 2: Class A vs Class C
- SVM 3: Class B vs Class C
→ Each SVM "votes", winner takes all (majority vote)

> **Sklearn's SVC uses One-vs-One by default** (generally more accurate but slower for many classes)

In [ ]:
# 3-class example using Iris
iris = datasets.load_iris()
X_ir = iris.data[:, :2]   # Only 2 features for visualization
y_ir = iris.target

sc_ir = StandardScaler()
X_ir_s = sc_ir.fit_transform(X_ir)
X_ir_tr, X_ir_te, y_ir_tr, y_ir_te = train_test_split(X_ir_s, y_ir, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
COLORS3 = ['#4C72B0', '#DD8452', '#55A868']
CMAP3 = ListedColormap(['#AEC6E8', '#F5C5A3', '#A8D8B9'])

for ax, kern, title in zip(axes,
                            ['linear', 'rbf', 'poly'],
                            ['Linear Kernel', 'RBF Kernel', 'Poly Kernel (d=3)']):
    kw = {} if kern == 'linear' else ({'gamma':'scale'} if kern=='rbf' else {'degree':3,'coef0':1,'gamma':'scale'})
    clf = SVC(kernel=kern, C=1, **kw)
    clf.fit(X_ir_tr, y_ir_tr)
    acc = accuracy_score(y_ir_te, clf.predict(X_ir_te))

    xx, yy = np.meshgrid(np.linspace(X_ir_s[:,0].min()-0.5, X_ir_s[:,0].max()+0.5, 300),
                         np.linspace(X_ir_s[:,1].min()-0.5, X_ir_s[:,1].max()+0.5, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP3)
    ax.contour(xx, yy, Z, colors='gray', linewidths=1, alpha=0.5)

    for cls, color, name in zip([0,1,2], COLORS3, iris.target_names):
        mask = y_ir==cls
        ax.scatter(X_ir_s[mask,0], X_ir_s[mask,1], c=color, s=50, edgecolors='k',
                   alpha=0.8, zorder=3, label=name)
    ax.set_title(f'{title}\n3 Classes | Test Acc={acc:.1%}', fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('SVM Multi-Class Classification — Iris Dataset (3 Classes)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 SVM handles multi-class via One-vs-One (OVO) by default.")
print(f"   3 classes → 3 binary SVMs trained (C(3,2) = 3 pairs)")
print(f"   10 classes → 45 binary SVMs trained (C(10,2) = 45 pairs)")
print("   For many classes (>10), consider LinearSVC which is much faster.")

---
# 📈 Part 16 — SVM for Regression (SVR)

SVM doesn't just classify — it can also **predict continuous numbers** (regression)!

This is called **SVR — Support Vector Regression**.

### The Idea (reversed logic):
- In classification: the margin separates classes, points outside are penalized
- In regression: we build a "tube" around the prediction line, points **inside** the tube are fine, points **outside** are penalized

### New Parameter: Epsilon (ε)
**ε (epsilon)** controls the width of the tube:
- Large ε → wide tube → more points inside → simpler model (may underfit)
- Small ε → narrow tube → fewer points inside → more complex model (may overfit)

Think of it as: *"How much error (deviation) am I willing to tolerate?"*

In [ ]:
# Generate noisy sine wave data
np.random.seed(0)
X_svr = np.sort(np.random.uniform(0, 6, 100)).reshape(-1, 1)
y_svr = np.sin(X_svr).ravel() + np.random.normal(0, 0.3, 100)

X_plot = np.linspace(0, 6, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epsilons = [0.05, 0.3, 1.0]
labels   = ['ε=0.05\n(very narrow tube)', 'ε=0.3\n(moderate ✅)', 'ε=1.0\n(very wide tube)']

for ax, eps, lbl in zip(axes, epsilons, labels):
    svr = SVR(kernel='rbf', C=1, epsilon=eps, gamma='scale')
    svr.fit(X_svr, y_svr)
    y_pred_svr = svr.predict(X_plot)

    ax.scatter(X_svr, y_svr, c='gray', s=25, alpha=0.5, label='Data')
    ax.plot(X_plot, y_pred_svr, 'b-', lw=2.5, label='SVR prediction')
    ax.fill_between(X_plot.ravel(),
                    y_pred_svr - eps, y_pred_svr + eps,
                    alpha=0.25, color='blue', label=f'ε-tube (±{eps})')
    ax.plot(X_plot, np.sin(X_plot), 'g--', lw=1.5, alpha=0.6, label='True sin(x)')

    # Highlight support vectors
    ax.scatter(X_svr[svr.support_], y_svr[svr.support_],
               c='red', s=60, zorder=5, label=f'Support Vectors ({len(svr.support_)})')

    ax.set_title(f'SVR — {lbl}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('X'); ax.set_ylabel('y')

plt.suptitle('Support Vector Regression (SVR) — Effect of Epsilon', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 SVR Key Points:")
print("   ε (epsilon): controls tube width — tolerance for errors")
print("   C          : same as in SVC — controls penalty for points outside tube")
print("   Points INSIDE the tube: ignored (no penalty)")
print("   Points OUTSIDE the tube: penalized")
print("   Start with ε≈0.1 and tune from there.")

---
# ⚖️ Part 17 — Pros & Cons of SVM (With Examples)

No algorithm is perfect. Let's look at where SVM shines and where it struggles.

In [ ]:
pros_cons = {
    "✅ PROS": [
        ("Effective in high dimensions",
         "Text classification: 10,000-word vocabulary → 10,000 features. SVM handles this well."),
        ("Works well with small datasets",
         "Medical imaging: only 500 MRI scans available. SVM often outperforms deep learning here."),
        ("Robust to outliers",
         "Soft margin + support vector focus means outliers far from the boundary don't affect it much."),
        ("Versatile via kernels",
         "One algorithm, 4 kernels = handles linear, circular, polynomial, and complex patterns."),
        ("Global optimum guaranteed",
         "Unlike neural networks, SVM optimization is convex — always finds the global best solution."),
        ("Memory efficient",
         "After training, only support vectors are stored. A 10,000-point model might need just 50 SVs."),
    ],
    "❌ CONS": [
        ("Slow on large datasets",
         "1M samples? SVM training time grows as O(n²)–O(n³). Use LinearSVC or neural nets instead."),
        ("Requires feature scaling",
         "Forget to scale → terrible results. Decision tree? No scaling needed. SVM always needs it."),
        ("Hard to interpret",
         "You can't easily explain 'why' a prediction was made. Decision trees are much more explainable."),
        ("Sensitive to hyperparameters",
         "Wrong C or gamma → bad results. You must tune carefully with cross-validation."),
        ("No native probability output",
         "SVC(probability=True) exists but uses slow Platt scaling. Random forests give probabilities naturally."),
        ("Doesn't scale to many classes",
         "100-class problem → 4,950 binary SVMs (OVO). Neural networks scale to 1000+ classes easily."),
    ]
}

for section, items in pros_cons.items():
    print(f"\n{'='*60}")
    print(f" {section}")
    print(f"{'='*60}")
    for point, example in items:
        print(f"\n  📌 {point}")
        print(f"     Example: {example}")

In [ ]:
# ── Visual demonstration: SVM vs Decision Tree on noisy data ──────────────────
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
X_cmp, y_cmp = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, class_sep=1.0, random_state=42)
sc_cmp = StandardScaler()
X_cmp_s = sc_cmp.fit_transform(X_cmp)
X_cmp_tr, X_cmp_te, y_cmp_tr, y_cmp_te = train_test_split(
    X_cmp_s, y_cmp, test_size=0.3, random_state=42)

models = [
    (SVC(kernel='rbf', C=1, gamma='scale'),        'SVM (RBF)'),
    (DecisionTreeClassifier(max_depth=3),           'Decision Tree'),
    (RandomForestClassifier(n_estimators=100, random_state=42), 'Random Forest'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (clf, name) in zip(axes, models):
    clf.fit(X_cmp_tr, y_cmp_tr)
    tr_acc = accuracy_score(y_cmp_tr, clf.predict(X_cmp_tr))
    te_acc = accuracy_score(y_cmp_te, clf.predict(X_cmp_te))

    xx, yy = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=CMAP)
    ax.contour(xx, yy, Z, colors='k', linewidths=1.5)
    colors_cmp = [COLORS[0] if c==0 else COLORS[1] for c in y_cmp]
    ax.scatter(X_cmp_s[:,0], X_cmp_s[:,1], c=colors_cmp, s=40, edgecolors='k', alpha=0.6, zorder=3)
    ax.set_title(f'{name}\nTrain: {tr_acc:.1%} | Test: {te_acc:.1%}', fontweight='bold')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

plt.suptitle('SVM vs Decision Tree vs Random Forest — Same Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observations:")
print("  • SVM:           Smooth, curved boundary — good generalization")
print("  • Decision Tree: Rectangular boundaries — may not generalize as well")
print("  • Random Forest: Ensemble of trees — often matches or beats SVM")
print()
print("💡 SVM vs Random Forest: Both are strong. Choose based on:")
print("   - Dataset size: RF is faster on large data")
print("   - Interpretability: tree-based models win")
print("   - High-dimensional sparse data (text): SVM usually wins")

In [ ]:
# ── Demonstrate scaling impact ────────────────────────────────────────────────
print("=" * 55)
print("  PRO TIP: What happens if you DON'T scale for SVM?")
print("=" * 55)

# Create data with vastly different feature scales
np.random.seed(42)
X_scale_demo = np.hstack([
    np.random.randn(200, 1) * 1000,    # Feature 1: range ~-3000 to 3000
    np.random.randn(200, 1) * 0.01,    # Feature 2: range ~-0.03 to 0.03
])
y_scale_demo = (X_scale_demo[:, 0]/1000 + X_scale_demo[:, 1]/0.01 > 0).astype(int)

X_sd_tr, X_sd_te, y_sd_tr, y_sd_te = train_test_split(
    X_scale_demo, y_scale_demo, test_size=0.3, random_state=42)

# Without scaling
svm_noscale = SVC(kernel='rbf', C=1, gamma='scale')
svm_noscale.fit(X_sd_tr, y_sd_tr)
acc_noscale = accuracy_score(y_sd_te, svm_noscale.predict(X_sd_te))

# With scaling
sc_demo = StandardScaler()
X_sd_tr_s = sc_demo.fit_transform(X_sd_tr)
X_sd_te_s = sc_demo.transform(X_sd_te)
svm_scale = SVC(kernel='rbf', C=1, gamma='scale')
svm_scale.fit(X_sd_tr_s, y_sd_tr)
acc_scale = accuracy_score(y_sd_te, svm_scale.predict(X_sd_te_s))

print(f"  Without scaling: {acc_noscale:.2%}  ← TERRIBLE")
print(f"  With scaling   : {acc_scale:.2%}  ← GREAT")
print()
print("  Why? Feature 1 ranges ±3000, Feature 2 ranges ±0.03.")
print("  Without scaling, SVM completely IGNORES Feature 2.")
print("  Scaling equalizes them — SVM can use both features properly.")
print()
print("  ⚠️  ALWAYS scale before SVM. Without it, results are meaningless.")

---
# 🗂️ Part 18 — Complete Cheat Sheet & Quick Reference

In [ ]:
cheat_sheet = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                       SVM MASTER CHEAT SHEET                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  KERNELS & FORMULAS                                                          ║
║  ─────────────────                                                           ║
║  Linear  : K(xᵢ,xⱼ) = xᵢ·xⱼ                                               ║
║  RBF     : K(xᵢ,xⱼ) = exp(-γ||xᵢ-xⱼ||²)         ← DEFAULT, use first    ║
║  Poly    : K(xᵢ,xⱼ) = (γ·xᵢ·xⱼ + r)^d                                     ║
║  Sigmoid : K(xᵢ,xⱼ) = tanh(γ·xᵢ·xⱼ + r)                                  ║
║                                                                              ║
║  PARAMETERS                                                                  ║
║  ──────────                                                                  ║
║  C      : Regularization. Large=strict/overfit, Small=relaxed/underfit     ║
║           Start: C=1   Range to try: [0.01, 0.1, 1, 10, 100, 1000]        ║
║                                                                              ║
║  gamma  : Influence radius (RBF/poly). Large=short reach, Small=long reach ║
║           Start: gamma='scale'  Range: [0.001, 0.01, 0.1, 1, 10]          ║
║                                                                              ║
║  degree : Polynomial complexity. d=2 or d=3 usually                        ║
║                                                                              ║
║  epsilon: SVR tube width. Start: 0.1  Range: [0.01, 0.1, 0.5, 1.0]        ║
║                                                                              ║
║  KERNEL SELECTION GUIDE                                                      ║
║  ──────────────────────                                                      ║
║  Many features (text, genes)?           → Linear                            ║
║  Not sure? General purpose?             → RBF  ✅                           ║
║  Image data, polynomial patterns?       → Poly (d=2 or d=3)                ║
║  Neural-network-like behavior?          → Sigmoid (rarely needed)           ║
║                                                                              ║
║  GOLDEN RULES                                                                ║
║  ────────────                                                                ║
║  1. ALWAYS use StandardScaler before SVM                                    ║
║  2. Use Pipeline(scaler + SVM) to avoid data leakage                        ║
║  3. Start with RBF kernel, tune C & gamma                                   ║
║  4. Use GridSearchCV with 5-fold CV to find best params                     ║
║  5. Large dataset (>50K)? Use LinearSVC instead                             ║
║                                                                              ║
║  QUICK TEMPLATE                                                              ║
║  ──────────────                                                              ║
║  from sklearn.svm import SVC                                                ║
║  from sklearn.preprocessing import StandardScaler                           ║
║  from sklearn.pipeline import Pipeline                                      ║
║  from sklearn.model_selection import GridSearchCV                           ║
║                                                                              ║
║  pipe = Pipeline([                                                           ║
║      ('scaler', StandardScaler()),                                          ║
║      ('svm', SVC(kernel='rbf'))                                             ║
║  ])                                                                          ║
║  params = {'svm__C': [0.1,1,10], 'svm__gamma': [0.1,1,10]}                ║
║  grid = GridSearchCV(pipe, params, cv=5)                                    ║
║  grid.fit(X_train, y_train)                                                 ║
║  print(grid.best_params_, grid.score(X_test, y_test))                      ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(cheat_sheet)

In [ ]:
# ── Final visual summary ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')

summary_text = [
    (0.05, 0.92, '🧠 SVM CONCEPT MAP', 16, 'black', 'bold'),
    (0.05, 0.82, '📐 CORE IDEA',        13, '#1565C0', 'bold'),
    (0.05, 0.75, 'Find the hyperplane that MAXIMIZES the MARGIN between classes', 11, 'black', 'normal'),
    (0.05, 0.68, 'Only SUPPORT VECTORS define the boundary — rest are irrelevant', 11, 'black', 'normal'),

    (0.05, 0.58, '🎛️ PARAMETERS',       13, '#1B5E20', 'bold'),
    (0.05, 0.51, 'C (large → strict, small margin) | C (small → relaxed, wide margin)', 11, 'black', 'normal'),
    (0.05, 0.44, 'γ (large → short reach, wiggly) | γ (small → long reach, smooth)', 11, 'black', 'normal'),
    (0.05, 0.37, 'degree d (polynomial complexity)  |  ε (regression tube width)', 11, 'black', 'normal'),

    (0.05, 0.27, '🌟 KERNELS',           13, '#4A148C', 'bold'),
    (0.05, 0.20, 'Linear: K=xᵢ·xⱼ  |  RBF: K=exp(-γ||xᵢ-xⱼ||²)  |  Poly: K=(γxᵢxⱼ+r)^d  |  Sig: K=tanh(γxᵢxⱼ+r)', 10, 'black', 'normal'),

    (0.05, 0.10, '🚀 WORKFLOW: Scale → Choose kernel → Tune C/γ → Evaluate → Pipeline',
     11, '#B71C1C', 'bold'),
]

for x, y, txt, sz, col, wt in summary_text:
    ax.text(x, y, txt, transform=ax.transAxes, fontsize=sz,
            color=col, fontweight=wt, va='center')

ax.add_patch(plt.Rectangle([0.01, 0.01], 0.98, 0.98,
                             fill=False, edgecolor='#cccccc', lw=2,
                             transform=ax.transAxes))

plt.title('SVM Complete Lecture — Summary', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print()
print("🎉 Congratulations! You've completed the full SVM Master Lecture!")
print()
print("Recommended next steps:")
print("  1. Apply this to your own dataset")
print("  2. Try sklearn.svm.LinearSVC for text classification")
print("  3. Explore sklearn.svm.SVR for regression problems")
print("  4. Compare SVM with Random Forest and Gradient Boosting")
print("  5. Read the sklearn SVM docs: https://scikit-learn.org/stable/modules/svm.html")